In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.pyplot import xticks
from matplotlib.pyplot import yticks
plt.style.use('ggplot')
#Importing DataSet
df=pd.read_csv("Cars Data.csv")
r,c=df.shape
print(f"Number of Rows:{r}")
print(f"Number of Columns:{c}")
print("\n")

#Calculating Statistics For data
stat=df.describe()
print(f"Statistics of The Numerical Data:{stat}")
print("\n")

#Dropping The Null Columns
drop=df.drop(columns=["Unnamed: 15","Unnamed: 16"])
#Dropping The Rows that have null values in column
df_clean=drop.dropna().copy()
print(f"Table After Dropping THE  Null Values:{df_clean}")
print(f"Number of Rows:{df_clean.shape[0]}")
print(f"Number of Columns:{df_clean.shape[1]}")
print("\n")

#Checking the Null Value
null=df_clean.isnull().sum()
print(f"Null Values:{null}")

#Shape of DataSet
column=df_clean.columns
print(f"Some Columns are:{column}")
c=df_clean.shape
print(f"Shape of DataSet:{c}")
print("\n")

#Data Info
info=df_clean.info()
print(f"Some Information of DataSet:{info}")
print("\n")

#Keeping The Neccessary Columns Only
type_count=df_clean['Type'].value_counts()
print(f"Counting The Types of Car:{type_count}")
print("\n")

#Transforming the MSRP Special Character into Numeric Value
df_clean["MSRP"]=df_clean["MSRP"].str.replace("$","", regex=False).str.replace(",","",regex=False).astype(float)
print(f"Numeric Values of MSRP is:{df_clean["MSRP"]}")
print("\n")
col_keep=["MSRP","Type","Origin","EngineSize","Horsepower","Weight"]
df_col=df_clean[col_keep]
df_final=pd.get_dummies(df_col,columns=["Type","Origin"],drop_first=True)
print("Final Columns For ML")
print(df_final.head())
print("\n")
print(df_final.info())
print("\n")
num=["MSRP","EngineSize","Horsepower","Weight"]
sns.pairplot(df_final[num],height=1.3,aspect=1.1)
plt.show()

#Correlation of an dataset
corr=df_final.corr()
print(f"Correlation of an DataSet:{corr}")

#Visualizing Using Heatmap
sns.heatmap(corr,annot=True,cmap="coolwarm",linewidth=0.5,fmt=".1f",annot_kws={"size":9})
plt.tight_layout()
plt.xticks(fontsize=7)
plt.yticks(fontsize=7)
plt.show()
print("\n")

#Dropping The Unneccessary Items
df_final1=df_final.drop(["Type_SUV","Type_Truck"],axis=1)
print(f"The Final Columns of The DataSet After Correlation:{df_final1}")
sns.heatmap(df_final1.corr(),annot=True,cmap="coolwarm",linewidth=0.5,fmt=".1f",annot_kws={"size":9})
plt.tight_layout()
plt.xticks(fontsize=7)
plt.yticks(fontsize=7)
plt.show()
print("\n")

#Traning and Testing The Data
from sklearn.model_selection import train_test_split
df_train,df_test=train_test_split(df_final1,train_size=0.80,test_size=0.20,random_state=0)


#Starting The Linear Regression
X_train=df_train[['EngineSize','Horsepower','Weight','Type_Sedan','Type_Sports','Type_Wagon','Origin_Europe','Origin_USA']]
y_train=df_train['MSRP'].astype('int')
X_test=df_test[['EngineSize','Horsepower','Weight','Type_Sedan','Type_Sports','Type_Wagon','Origin_Europe','Origin_USA']]
y_test=df_test['MSRP'].astype('int')

print(f"Training The DataSet of X:\n{X_train}")
print(f"Training The DataSet of Y:\n{y_train}")
print("\n")
print(f"Testing The DataSet of X:\n{X_test}")
print("\n")

#Scaling The Data
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_Scale=scaler.fit_transform(X_train)
X_test_Scale=scaler.transform(X_test)

#Learning The Model(Linear Regression)
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
model=lr.fit(X_train_Scale,y_train)

#For Polynomial Regression
from sklearn.preprocessing import PolynomialFeatures
poly=PolynomialFeatures(degree=2)
x_poly=poly.fit_transform(X_train_Scale)
pf=LinearRegression()
poly_reg=pf.fit(x_poly,y_train)

data=df_final1.iloc[400:401]
actual_price=data['MSRP']
data_drop=data.drop(['MSRP'],axis=1)
data_scale=scaler.transform(data_drop)
print("\n")
print(f"Data For Testing:\n{data_scale}")
print("\n")

#Predicting the Regression Model
print("Linear Regression:")
print(f"Predicting The Car Price:{model.predict(data_scale)[0]:,.2f}")
print(f"Actual Price:{actual_price}")
print("\n")

#Predicting The Polynomial Model
print("Polynomial Regression:")
predict_price=poly_reg.predict(poly.transform(data_scale))
print(f"Predicting The MSRP in Polynomial:{predict_price[0]:,.2f}")
print(f"Actual MSRP :{actual_price}")
print("\n")

#Finding the Score of The Traning Data
print("Predicting The Score of That DataSet:")
print(f"Score of Linear Regression:{model.score(X_train_Scale,y_train)*100:.2f}")
print(f"Score of Polynomial Regression:{poly_reg.score(x_poly,y_train)*100:.2f}")
print("\n")

#Finding The Score of Testing Data
print("Predicting The Score of The Testing Data:")
print(f"Score of Linear Regression:{model.score(X_test_Scale,y_test)*100:.2f}")
print(f"Score of Polynomial Regression:{poly_reg.score(poly.transform(X_test_Scale),y_test)*100:.2f}")

#Importing The Pickle
import pickle as pk
file_name="Car-Prediction.pkl"
pk.dump(model,open(file_name,"wb"))

#Asking The input to the user
print("Car-Price-Prediction:\n")
eng=int(input("Enter Your Engine-Size:"))
hor=int(input("Enter Your Horse-Power:"))
wei=int(input("Enter Your Car Weight:"))
sedan=int(input("Enter 1 if you want Sedan otherwise Enter 0:"))
sport=int(input("Enter 1 if you want Sport otherwise Enter 0:"))
wagon=int(input("Enter 1 if you want Wagon otherewise Enter 0:"))
euro=int(input("Enter 1 if your Origin is Euro: otherwise Enter 0:"))
usa=int(input("Enter 1 if your  Origin is USA: otherwise Enter 0:"))
print("\n")

df1=pd.DataFrame({"EngineSize":[eng],"Horsepower":[hor],"Weight":[wei],"Type_Sedan":[sedan],"Type_Sports":[sport],"Type_Wagon":[wagon],
                  "Origin_Europe":[euro],"Origin_USA":[usa]})

df1_scale=scaler.transform(df1)

#Predicting The User Data
predict_price=model.predict(df1_scale)
print(f"Predicted Price of Your Car is :{predict_price[0]:.2f}")

                  

      















Number of Rows:428
Number of Columns:17


Statistics of The Numerical Data:       EngineSize   Cylinders  Horsepower    MPG_City  MPG_Highway  \
count  428.000000  426.000000  428.000000  428.000000   428.000000   
mean     3.196729    5.807512  215.885514   20.060748    26.843458   
std      1.108595    1.558443   71.836032    5.238218     5.741201   
min      1.300000    3.000000   73.000000   10.000000    12.000000   
25%      2.375000    4.000000  165.000000   17.000000    24.000000   
50%      3.000000    6.000000  210.000000   19.000000    26.000000   
75%      3.900000    6.000000  255.000000   21.250000    29.000000   
max      8.300000   12.000000  500.000000   60.000000    66.000000   

            Weight   Wheelbase      Length  Unnamed: 15  Unnamed: 16  
count   428.000000  428.000000  428.000000          0.0          0.0  
mean   3577.953271  108.154206  186.362150          NaN          NaN  
std     758.983215    8.311813   14.357991          NaN          NaN  
min    185